# TP2 : topics et messages

| Repère | Valeur |
|---|---|
| Moment | Jour 1, 11:50-12:30, après la démo du TP (11:45-11:50) |
| Compétences | C4 Administrer les topics ; C5 Produire et consommer |
| Slides de rappel | S6 (topics et partitions), S7 (offsets et ordre), S8 (partitionnement avec et sans clé), S22 (outils de `bin/`) |
| Version Markdown | `enonce.md`, mêmes étapes en ligne de commande seule |

**Mode d'emploi du notebook**

- Exécutez les cellules dans l'ordre avec `Maj+Entrée`. Après un redémarrage du noyau, les variables sont perdues : relancez les trois cellules du début (connexion, fonctions d'aide, outils du TP), puis les cellules « Code fourni » des étapes déjà faites. Ne relancez pas une cellule qui modifie le cluster (création d'un topic, envoi de messages) : elle échouerait, ou referait l'action.
- Chaque étape suit le même schéma qu'au TP1 : une explication, un **exemple en ligne de commande**, une cellule **`# TODO`**, puis une **vérification**.
- **Vous n'avez pas de code à écrire.** Dans une cellule `# TODO`, remplacez chaque `...` par une valeur : un **réglage** (`"murmur2_random"`, une durée en millisecondes...) ou votre **prévision** du résultat (un nombre, `True` ou `False`). Les noms qui commencent par `PREDICTED_` sont des prévisions : décidez-les avant d'exécuter, la vérification les compare au résultat.
- Sous les réponses, le code est fourni : lisez ses commentaires, ne le modifiez pas. Tant qu'il reste un `...`, la ligne `if filled(...):` l'empêche de s'exécuter et affiche « À compléter ».
- Quand la commande modifie le cluster (création d'un topic, envoi de messages), son équivalent en ligne de commande est affiché **sans être exécuté**, avec la mention « ne pas lancer en plus du TODO » : le lancer en plus doublerait l'opération et fausserait les vérifications.
- Les appels du client d'administration (`admin = AdminClient(CONF)`, créé dans chaque cellule qui en a besoin) renvoient un dictionnaire de **futurs** : `{name: future}`. `future.result()` attend la réponse du cluster et lève `KafkaException` en cas d'échec ; `e.args[0].name()` donne alors le code d'erreur.
- Ce TP utilise aussi le producteur (`Producer`) et le consommateur (`Consumer`) de `confluent-kafka`.

In [ ]:
# Connexion au lab : adresses des brokers, configuration du client Python, dossier de travail
import os
import time

from confluent_kafka.admin import AdminClient

BS = os.environ["BS"]  # listener INTERNAL, comme $BS dans le conteneur tools
CONF = {"bootstrap.servers": BS, "log_level": 2}

# Les commandes docker compose et les fichiers config/ se lisent depuis le dossier lab/
LAB_DIR = os.environ["FORMATION_DIR"] + "/lab"
os.chdir(LAB_DIR)
os.environ["COMPOSE_PROGRESS"] = "plain"  # docker compose sans animation dans le notebook

In [ ]:
# Trois fonctions d'aide, écrites ici : elles ne font pas partie de confluent-kafka
def wait_for(condition, timeout=60, step=2, message="condition"):
    """Rappelle condition() jusqu'à obtenir une valeur vraie, et la renvoie."""
    start = time.time()
    while True:
        try:
            value = condition()
        except Exception:
            value = None
        if value:
            return value
        if time.time() - start >= timeout:
            raise TimeoutError("%s : non atteinte en %d s" % (message, timeout))
        time.sleep(step)

def table(lines, headers):
    """Affiche des lignes en tableau Markdown dans le notebook."""
    from IPython.display import Markdown, display

    md = ["| " + " | ".join(headers) + " |", "|" + "---|" * len(headers)]
    md += ["| " + " | ".join(str(v).replace("|", "\\|") for v in line) + " |" for line in lines]
    display(Markdown("\n".join(md)))

def filled(*answers):
    """Vrai si toutes les réponses d'une cellule TODO sont données ; sinon affiche « À compléter »."""
    for answer in answers:
        if answer is ...:
            print("À compléter : remplacez chaque ... de la cellule par votre réponse, puis exécutez-la de nouveau.")
            return False
    return True

In [ ]:
# Outils du TP : imports du client Kafka
import re

from confluent_kafka import Consumer, KafkaException, Producer, TopicCollection, TopicPartition
from confluent_kafka.admin import (AlterConfigOpType, ConfigEntry, ConfigResource, ConfigSource,
                                   NewPartitions, NewTopic, OffsetSpec)

print("Bootstrap INTERNAL :", BS)

## 1. Objectifs

À la fin du TP, vous savez :

- constater que la création automatique des topics est désactivée, et pourquoi ;
- créer, décrire, modifier et supprimer un topic ;
- lire le placement des répliques : leader, répliques, ISR ;
- produire des messages avec clé depuis un fichier, et sans clé ;
- lire une partition à partir d'un offset donné ;
- vérifier que tous les messages d'une même clé arrivent dans la même partition ;
- mesurer l'effet d'un ajout de partitions sur la répartition des clés ;
- poser, lire et retirer une configuration de topic.

## 2. Prérequis

- TP1 terminé : lab démarré et `scripts/validate.sh` au vert.
- Jeu de données `lab/data/orders.txt` : 30 commandes au format `client|commande JSON`, clients `C001` à `C007`. Il est lisible depuis le notebook (`"data/orders.txt"`) et depuis `tools` (`/opt/lab/data/orders.txt`).

## 3. Architecture utilisée

```
  jupyter (client Python) et tools (CLI) ──> kafka-1, kafka-2, kafka-3
                                             (listener INTERNAL, port 19092, BS)

  Topic orders, 3 partitions, RF 3 : chaque partition a 1 leader et 2 followers,
  répartis sur les 3 brokers.

            kafka-1          kafka-2          kafka-3
  orders-0  follower         LEADER           follower
  orders-1  follower         follower         LEADER
  orders-2  LEADER           follower         follower
```

Le placement exact des leaders varie d'un poste à l'autre.

## 4. Contexte métier

Le service Commandes de NovaShop va publier chaque commande dans un topic `orders`. Exigence métier : les commandes d'un même client doivent être traitées dans l'ordre. L'équipe Data veut aussi un topic `clickstream` pour la navigation, conservé 24 heures.

Vous préparez ces topics et vérifiez que la clé de partitionnement garantit bien l'ordre par client. La fiche des topics affichée en fin de notebook résume ce que vous avez livré.

## 5. Énoncé

1. Constatez que produire vers un topic qui n'existe pas échoue.
2. Créez `orders` (3 partitions, facteur de réplication 3) et alimentez-le avec `orders.txt`, en utilisant l'identifiant client comme clé.
3. Prouvez que toutes les commandes d'un même client sont dans la même partition.
4. Passez `orders` à 6 partitions, alimentez-le à nouveau et mesurez ce que l'opération change pour l'ordre par client.
5. Créez `clickstream` (6 partitions) avec une rétention de 24 heures.

## 6. Étapes

### 6.1 Produire vers un topic absent

Le lab fixe `auto.create.topics.enable=false` sur chaque nœud : un producteur ne peut pas créer un topic en y écrivant. On le constate avec l'outil Java, puis avec le client Python.

**Exemple en ligne de commande** (le producteur abandonne au bout de 5 s) :

In [ ]:
!grep auto.create config/kafka/kafka-1/server.properties
!docker exec kafka-tools bash -c "echo 'test' | kafka-console-producer.sh --bootstrap-server $BS --topic orders --command-property max.block.ms=5000"

In [ ]:
# TODO 6.1 : remplacez le ... par votre prévision, puis exécutez (5 secondes environ)
PREDICTED_CREATED = ...  # l'envoi d'un message vers orders, qui n'existe pas, va-t-il créer le topic ?
                         # (True ou False)

# Code fourni, à lire : rien à modifier en dessous
errors = []
if filled(PREDICTED_CREATED):
    # Ce producteur déclare un topic inconnu au bout de 5 s (30 s par défaut)
    test_producer = Producer({"bootstrap.servers": BS, "topic.metadata.propagation.max.ms": 5000, "log_level": 4})

    def delivery_report(err, msg):
        """Rapport de livraison : appelé une fois par message, pendant flush(), jamais pendant produce()."""
        errors.append(err)  # err vaut None si le message est écrit

    test_producer.produce("orders", value="test", on_delivery=delivery_report)
    test_producer.flush(15)  # attend le rapport de livraison, 15 s au plus

print(errors)

In [ ]:
# Vérification 6.1
admin = AdminClient(CONF)
assert len(errors) == 1 and errors[0] is not None, "À revoir : un rapport de livraison, en erreur"
print("OK  un rapport de livraison, en erreur")
assert errors[0].name() == "UNKNOWN_TOPIC_OR_PART", "À revoir : code d'erreur : %s" % errors[0].name()
print("OK  code d'erreur : %s" % errors[0].name())
created = admin.list_topics("orders", timeout=10).topics["orders"].error is None
assert not created, "À revoir : le topic orders n'a pas été créé par l'envoi"
print("OK  le topic orders n'a pas été créé par l'envoi")
assert PREDICTED_CREATED == created, "À revoir : prévision %r : la création automatique est désactivée dans le lab" % (PREDICTED_CREATED,)
print("OK  prévision juste : topic non créé")

**Résultat attendu** :

- en ligne de commande, des avertissements `{orders=UNKNOWN_TOPIC_OR_PARTITION}`, puis au bout de 5 s `TimeoutException: Topic orders not present in metadata after 5000 ms.` ;
- en Python, au bout de 5 s, un rapport de livraison en erreur : `KafkaError{code=UNKNOWN_TOPIC_OR_PART,val=3,str="Broker: Unknown topic or partition"}`.

### 6.2 Créer le topic `orders`

Un topic se crée avec un nombre de partitions et un facteur de réplication (RF). Le RF ne peut pas dépasser le nombre de brokers.

**Exemple en ligne de commande** : demande d'un RF de 4 sur un cluster de 3 brokers.

In [ ]:
!docker exec kafka-tools kafka-topics.sh --bootstrap-server $BS --create --topic orders-rf4 --partitions 1 --replication-factor 4

Équivalent en ligne de commande de la création de `orders` (**ne pas lancer en plus du TODO**) :

```bash
# [tools]
kafka-topics.sh --bootstrap-server $BS --create --topic orders --partitions 3 --replication-factor 3
```

In [ ]:
# TODO 6.2 : remplacez les trois ... puis exécutez UNE SEULE FOIS
PARTITIONS = ...          # nombre de partitions de orders (point 5 de l'énoncé)
REPLICATION_FACTOR = ...  # facteur de réplication de orders (point 5 de l'énoncé)
PREDICTED_RF4_OK = ...    # un facteur de réplication de 4, sur ce cluster de 3 brokers,
                          # sera-t-il accepté ? (True ou False)

# Code fourni, à lire : rien à modifier en dessous
code_rf4 = None
if filled(PARTITIONS, REPLICATION_FACTOR, PREDICTED_RF4_OK):
    admin = AdminClient(CONF)
    # create_topics() renvoie {nom: futur} ; result() attend la création
    futures = admin.create_topics([NewTopic("orders", num_partitions=PARTITIONS, replication_factor=REPLICATION_FACTOR)])
    futures["orders"].result()
    print("Topic orders créé")

    # validate_only=True : le cluster valide la demande sans rien créer
    try:
        admin.create_topics([NewTopic("orders-rf4", num_partitions=1, replication_factor=4)],
                            validate_only=True)["orders-rf4"].result()
    except KafkaException as e:
        code_rf4 = e.args[0].name()  # le code d'erreur renvoyé par le cluster
        print(e.args[0].str())       # le message d'erreur complet

print("code_rf4 :", code_rf4)

In [ ]:
# Vérification 6.2
admin = AdminClient(CONF)
meta_orders = admin.list_topics("orders", timeout=10).topics["orders"]
assert meta_orders.error is None, "À revoir : topic orders créé"
print("OK  topic orders créé")
assert len(meta_orders.partitions) == 3, "À revoir : 3 partitions"
print("OK  3 partitions")
assert code_rf4 == "INVALID_REPLICATION_FACTOR", "À revoir : RF 4 refusé : %s" % code_rf4
print("OK  RF 4 refusé : %s" % code_rf4)
assert PREDICTED_RF4_OK is False, "À revoir : prévision %r : le RF ne peut pas dépasser le nombre de brokers" % (PREDICTED_RF4_OK,)
print("OK  prévision juste : RF 4 refusé")

**Résultat attendu** : `orders` est créé. Le facteur de réplication de 4 est refusé, en ligne de commande comme en Python :

```
Unable to replicate the partition 4 time(s): The target replication factor of 4 cannot be reached because only 3 broker(s) are registered or some brokers have all their log directories cordoned.
```

### 6.3 Décrire le topic

Chaque partition a un leader, une liste de répliques et un ISR (répliques synchronisées). La première réplique de la liste est le **leader préféré**.

**Exemple en ligne de commande** :

In [ ]:
!docker exec kafka-tools kafka-topics.sh --bootstrap-server $BS --describe --topic orders

In [ ]:
# TODO 6.3 : remplacez le ... par votre prévision, puis exécutez
PREDICTED_ISR_SIZE = ...  # sur un topic tout juste créé, combien de répliques chaque partition
                          # a-t-elle dans son ISR ? (un nombre)

# Code fourni, à lire : rien à modifier en dessous
placement = []
if filled(PREDICTED_ISR_SIZE):
    admin = AdminClient(CONF)
    description = admin.describe_topics(TopicCollection(["orders"]))["orders"].result()
    # Une ligne par partition : numéro, leader, répliques, ISR (identifiants des nœuds)
    placement = [(p.id, p.leader.id, [r.id for r in p.replicas], [r.id for r in p.isr])
                 for p in description.partitions]
    print("TopicId :", description.topic_id)

table(placement, ["partition", "leader", "répliques", "isr"])

In [ ]:
# Vérification 6.3
assert len(placement) == 3, "À revoir : 3 partitions décrites"
print("OK  3 partitions décrites")
assert all(len(rep) == 3 and set(isr) == set(rep) for _, _, rep, isr in placement), "À revoir : 3 répliques par partition, toutes dans l'ISR"
print("OK  3 répliques par partition, toutes dans l'ISR")
assert all(leader == rep[0] for _, leader, rep, _ in placement), "À revoir : chaque leader est le leader préféré"
print("OK  chaque leader est le leader préféré")
assert sorted(p[1] for p in placement) == [1, 2, 3], "À revoir : leaders répartis sur les 3 brokers"
print("OK  leaders répartis sur les 3 brokers")
assert PREDICTED_ISR_SIZE == len(placement[0][3]), "À revoir : prévision %r, %d répliques dans l'ISR" % (PREDICTED_ISR_SIZE, len(placement[0][3]))
print("OK  prévision juste : %d répliques dans l'ISR" % len(placement[0][3]))

**Résultat attendu** : 3 partitions, chacune avec 3 répliques toutes dans l'ISR, et des leaders répartis sur les 3 brokers. En ligne de commande, la ligne de titre affiche aussi des configurations : `min.insync.replicas=1`, `segment.bytes=1073741824`, `unclean.leader.election.enable=false`.

### 6.4 Produire avec clé

Chaque ligne de `orders.txt` est de la forme `client|commande JSON`. L'identifiant client sert de clé : le producteur calcule la partition à partir d'un hachage de la clé, donc toutes les commandes d'un client vont dans la même partition.

**Attention au partitionneur.** Les clients Java, dont les outils `kafka-*.sh`, utilisent le hachage murmur2. librdkafka, la bibliothèque du client Python, utilise par défaut un autre hachage (CRC32, partitionneur `consistent_random`) : la même clé irait dans une autre partition. Le réglage `partitioner=murmur2_random` aligne le client Python sur Java. Il est indispensable dès que des producteurs écrits dans plusieurs langages alimentent le même topic.

**Exemple en ligne de commande** :

In [ ]:
!docker exec kafka-tools head -3 /opt/lab/data/orders.txt

Équivalent en ligne de commande de l'envoi (**ne pas lancer en plus du TODO**) :

```bash
# [tools]
kafka-console-producer.sh --bootstrap-server $BS --topic orders \
  --reader-property parse.key=true --reader-property 'key.separator=|' \
  < /opt/lab/data/orders.txt
```

**Exécutez la cellule `TODO` une seule fois** : chaque exécution renvoie les 30 commandes. En cas de doublon, voir le point 12, erreurs fréquentes.

**Code fourni** : `produce_orders()` envoie les 30 lignes du fichier avec le producteur `producer` créé dans le `TODO`. Elle resservira à l'étape 6.7.

In [ ]:
# Code fourni : envoyer les 30 commandes de orders.txt, le client comme clé
def produce_orders():
    """Envoie orders.txt dans orders avec le producteur courant ; renvoie le nombre de messages confirmés."""
    confirmed = []

    def delivery_report(err, msg):
        # Appelée une fois par message, pendant flush() : err vaut None si le message est écrit
        if err is None:
            confirmed.append(msg)

    with open("data/orders.txt") as f:
        for line in f.read().splitlines():
            # "C002|{...}" : coupure au premier "|", le client est la clé, la commande JSON la valeur
            key, value = line.split("|", 1)
            producer.produce("orders", key=key, value=value, on_delivery=delivery_report)
    # produce() ne fait que mettre en file : flush() envoie et attend les rapports de livraison
    producer.flush(30)
    return len(confirmed)

In [ ]:
# TODO 6.4 : remplacez le ... puis exécutez UNE SEULE FOIS (chaque exécution renvoie les 30 commandes)
PARTITIONER = ...  # partitionneur du client Python : "consistent_random" (défaut de librdkafka)
                   # ou "murmur2_random" (même hachage que les outils Java) ? Relisez l'encadré ci-dessus.

# Code fourni, à lire : rien à modifier en dessous
sent = 0
if filled(PARTITIONER):
    if PARTITIONER != "murmur2_random":
        # Protection du TP : les clés iraient dans d'autres partitions qu'avec les outils Java,
        # et orders serait faussé pour la journée. Rien n'est envoyé.
        print("Rien n'est envoyé : avec ce partitionneur, une même clé n'irait pas dans la même partition "
              "qu'avec les outils Java. Relisez l'encadré « Attention au partitionneur ».")
    else:
        producer = Producer({"bootstrap.servers": BS, "partitioner": PARTITIONER, "log_level": 4})
        sent = produce_orders()

print(sent, "messages confirmés")

In [ ]:
# Vérification 6.4
assert sent == 30, "À revoir : 30 messages confirmés par le cluster (%s)" % sent
print("OK  30 messages confirmés par le cluster (%s)" % sent)
ends_cli = !docker exec kafka-tools kafka-get-offsets.sh --bootstrap-server $BS --topic orders 2>&1
ends_cli = ends_cli.n
total = sum(int(n) for n in re.findall(r"^orders:\d+:(\d+)$", ends_cli, re.M))
assert total == 30, "À revoir : 30 messages dans orders (%d) : envoi fait une seule fois" % total
print("OK  30 messages dans orders (%d) : envoi fait une seule fois" % total)

**Résultat attendu** : `30 messages confirmes`. La première ligne du fichier est `C002|{"order_id":"O-1001","customer":"C002","amount":15.03,"city":"Lille"}`.

### 6.5 Lire les offsets et consommer

L'offset de fin d'une partition est l'offset du prochain message à écrire. Tant que rien n'a été supprimé, il est égal au nombre de messages de la partition.

Pour lire une partition précise à partir d'un offset, le consommateur utilise `assign()` : il lit directement, sans groupe actif ni rééquilibrage (les groupes sont le sujet du TP3).

**Exemple en ligne de commande** :

In [ ]:
!docker exec kafka-tools kafka-get-offsets.sh --bootstrap-server $BS --topic orders
!docker exec kafka-tools kafka-console-consumer.sh --bootstrap-server $BS --topic orders --partition 0 --offset 0 --max-messages 5 --timeout-ms 10000 --formatter-property print.key=true --formatter-property print.partition=true --formatter-property print.offset=true

**Code fourni** : `end_offsets()` donne l'offset de fin de chaque partition, comme `kafka-get-offsets.sh` ; `read()` lit une partition à partir d'un offset, comme `--partition` et `--offset` de `kafka-console-consumer.sh`. Elles resserviront à l'étape 6.6 et au TP3.

In [ ]:
# Code fourni : offsets de fin d'un topic, et lecture d'une partition à partir d'un offset
def end_offsets(topic):
    """{partition: offset de fin} : l'offset que recevra le prochain message de chaque partition."""
    admin = AdminClient(CONF)
    # Nombre de partitions du topic, lu dans les métadonnées du cluster
    num_partitions = len(admin.list_topics(topic, timeout=10).topics[topic].partitions)
    # Une seule requête pour toutes les partitions ; la réponse contient un futur par partition
    futures = admin.list_offsets({TopicPartition(topic, p): OffsetSpec.latest() for p in range(num_partitions)})
    # f.result() attend la réponse du broker ; les offsets sont rangés par numéro de partition
    return {tp.partition: f.result().offset for tp, f in sorted(futures.items(), key=lambda x: x[0].partition)}


def read(topic, partition, start, count):
    """Lit au plus count messages de la partition, à partir de l'offset start."""
    # Sans validation d'offsets, le groupe tp2-lecture n'est pas enregistré sur le cluster
    consumer = Consumer({"bootstrap.servers": BS, "group.id": "tp2-lecture",
                         "enable.auto.commit": False, "log_level": 4})
    # assign() : lecture directe de la partition, sans groupe actif ni rééquilibrage
    consumer.assign([TopicPartition(topic, partition, start)])
    messages = []
    while len(messages) < count:
        # Au plus les messages manquants ; liste vide si rien n'arrive en 10 s
        batch = consumer.consume(count - len(messages), timeout=10)
        if not batch:
            break
        messages += [m for m in batch if m.error() is None]
    consumer.close()
    return messages

In [ ]:
# TODO 6.5 : remplacez les deux ... puis exécutez
START_OFFSET = ...     # à partir de quel offset lire les 5 premiers messages de la partition 0 ? (un nombre)
PREDICTED_TOTAL = ...  # somme des offsets de fin des 3 partitions, après l'envoi de 6.4 ? (un nombre)

# Code fourni, à lire : rien à modifier en dessous
ends = {}
first_messages = []
if filled(START_OFFSET, PREDICTED_TOTAL):
    ends = end_offsets("orders")
    # Chaque message a les méthodes partition(), offset(), key() et value() (en octets : .decode())
    first_messages = [(m.partition(), m.offset(), m.key().decode()) for m in read("orders", 0, START_OFFSET, 5)]

print(ends)
table(first_messages, ["partition", "offset", "clé"])

In [ ]:
# Vérification 6.5
assert sum(ends.values()) == 30, "À revoir : 30 messages au total : %s" % ends
print("OK  30 messages au total : %s" % ends)
assert ends == {0: 6, 1: 8, 2: 16}, "À revoir : répartition identique à celle du producteur Java : %s" % ends
print("OK  répartition identique à celle du producteur Java : %s" % ends)
assert len(first_messages) == 5 and {p[2] for p in first_messages} == {"C002"}, "À revoir : 5 premiers messages de la partition 0, tous du client C002"
print("OK  5 premiers messages de la partition 0, tous du client C002")
assert PREDICTED_TOTAL == sum(ends.values()), "À revoir : prévision %r, total observé %d" % (PREDICTED_TOTAL, sum(ends.values()))
print("OK  prévision juste : %d messages" % sum(ends.values()))

**Résultat attendu** : `{0: 6, 1: 8, 2: 16}`, identique à `kafka-get-offsets.sh` (`orders:0:6`, `orders:1:8`, `orders:2:16`). Les 5 premiers messages de la partition 0 sont tous du client `C002`, aux offsets 0 à 4.

### 6.6 Défi : correspondance clé et partition

Prouvez que toutes les commandes d'un même client sont dans une seule partition.

**Exemple en ligne de commande** : le consommateur affiche la partition et la clé de chaque message. En ligne de commande, `awk` et `sort -u` feraient le reste ; ici, ce sera Python.

In [ ]:
!docker exec kafka-tools bash -c "kafka-console-consumer.sh --bootstrap-server $BS --topic orders --from-beginning --max-messages 5 --timeout-ms 10000 --formatter-property print.key=true --formatter-property print.partition=true 2>/dev/null"

**Code fourni** : `map_keys()` lit chaque partition en entier, avec `end_offsets()` et `read()`, et note dans quelles partitions apparaît chaque clé.

In [ ]:
# Code fourni : pour chaque clé, la liste des partitions où elle apparaît
def map_keys(topic):
    """Renvoie {clé: [partitions triées]} en lisant toutes les partitions du topic."""
    key_map = {}
    for partition, end in end_offsets(topic).items():
        # De l'offset 0 à l'offset de fin : rien n'a encore été supprimé de ce topic
        for m in read(topic, partition, 0, end):
            # setdefault() crée un ensemble vide pour une clé nouvelle ; add() y ajoute la partition
            key_map.setdefault(m.key().decode(), set()).add(partition)
    return {key: sorted(parts) for key, parts in key_map.items()}

In [ ]:
# TODO 6.6 (défi) : remplacez le ... par votre prévision, puis exécutez
PREDICTED_PARTITIONS_PER_CLIENT = ...  # dans combien de partitions trouvera-t-on les commandes
                                       # d'un même client ? (un nombre)

# Code fourni, à lire : rien à modifier en dessous
key_map = {}
if filled(PREDICTED_PARTITIONS_PER_CLIENT):
    key_map = map_keys("orders")

for client, partitions in sorted(key_map.items()):
    print(client, partitions)

In [ ]:
# Vérification 6.6
assert len(key_map) == 7, "À revoir : 7 clients trouvés"
print("OK  7 clients trouvés")
assert all(len(p) == 1 for p in key_map.values()), "À revoir : chaque client dans une seule partition"
print("OK  chaque client dans une seule partition")
assert key_map.get("C002") == [0], "À revoir : C002 en partition 0, comme avec le producteur Java"
print("OK  C002 en partition 0, comme avec le producteur Java")
assert PREDICTED_PARTITIONS_PER_CLIENT == 1, "À revoir : prévision %r : une clé va toujours dans la même partition" % (PREDICTED_PARTITIONS_PER_CLIENT,)
print("OK  prévision juste : une partition par client")

**Résultat attendu** : 7 lignes pour 7 clients, chacun dans une seule partition : `C002` en 0, `C006` et `C007` en 1, `C001`, `C003`, `C004` et `C005` en 2.

### 6.7 Ajouter des partitions

On peut augmenter le nombre de partitions d'un topic, jamais le réduire. Les messages existants restent où ils sont.

Un producteur déjà lancé garde en cache les métadonnées du topic : il ne voit les nouvelles partitions qu'au rafraîchissement suivant, jusqu'à 5 minutes plus tard (`topic.metadata.refresh.interval.ms` pour librdkafka, `metadata.max.age.ms` pour Java). Pour que le second envoi utilise les 6 partitions, recréez le producteur.

**Exemple en ligne de commande** : demande de réduction à 2 partitions.

In [ ]:
!docker exec kafka-tools kafka-topics.sh --bootstrap-server $BS --alter --topic orders --partitions 2

Équivalent en ligne de commande de l'ajout et du second envoi (**ne pas lancer en plus du TODO**) :

```bash
# [tools]
kafka-topics.sh --bootstrap-server $BS --alter --topic orders --partitions 6
kafka-console-producer.sh --bootstrap-server $BS --topic orders \
  --reader-property parse.key=true --reader-property 'key.separator=|' \
  < /opt/lab/data/orders.txt
```

In [ ]:
# TODO 6.7 : remplacez les deux ... par vos prévisions, puis exécutez UNE SEULE FOIS
PREDICTED_MOVED = ...      # après le passage à 6 partitions et le second envoi, certains clients
                           # auront-ils des commandes dans 2 partitions ? (True ou False)
PREDICTED_SHRINK_OK = ...  # le retour de 6 à 3 partitions sera-t-il accepté ? (True ou False)

# Code fourni, à lire : rien à modifier en dessous
sent_2, ends_2, key_map_2, shrink_code = 0, {}, {}, None
if filled(PREDICTED_MOVED, PREDICTED_SHRINK_OK):
    admin = AdminClient(CONF)
    # Passage à 6 partitions : les messages existants restent où ils sont
    admin.create_partitions([NewPartitions("orders", 6)])["orders"].result()
    # Le temps que les brokers diffusent les nouvelles métadonnées
    wait_for(lambda: len(admin.list_topics("orders", timeout=10).topics["orders"].partitions) == 6,
             timeout=30, message="6 partitions visibles")

    # Nouveau producteur : il lit les métadonnées à jour (6 partitions), puis second envoi
    producer = Producer({"bootstrap.servers": BS, "partitioner": "murmur2_random", "log_level": 4})
    sent_2 = produce_orders()

    ends_2 = end_offsets("orders")
    key_map_2 = map_keys("orders")

    # Demande de retour à 3 partitions : en cas de refus, le code d'erreur va dans shrink_code
    try:
        admin.create_partitions([NewPartitions("orders", 3)])["orders"].result()
    except KafkaException as e:
        shrink_code = e.args[0].name()
        print(e.args[0].str())

print(ends_2)
for client, partitions in sorted(key_map_2.items()):
    print(client, partitions)
print("code_reduction :", shrink_code)

In [ ]:
# Vérification 6.7
assert sent_2 == 30, "À revoir : 30 messages renvoyés"
print("OK  30 messages renvoyés")
assert ends_2 == {0: 12, 1: 16, 2: 21, 3: 0, 4: 0, 5: 11}, "À revoir : offsets de fin : %s" % ends_2
print("OK  offsets de fin : %s" % ends_2)
moved = sorted(c for c, p in key_map_2.items() if len(p) > 1)
assert moved == ["C001", "C003", "C004"], "À revoir : clients présents dans 2 partitions : %s" % moved
print("OK  clients présents dans 2 partitions : %s" % moved)
assert shrink_code == "INVALID_PARTITIONS", "À revoir : réduction refusée : %s" % shrink_code
print("OK  réduction refusée : %s" % shrink_code)
assert PREDICTED_MOVED == bool(moved), "À revoir : prévision %r : %d client(s) dans 2 partitions" % (PREDICTED_MOVED, len(moved))
print("OK  prévision juste : des clients ont changé de partition")
assert PREDICTED_SHRINK_OK == (shrink_code is None), "À revoir : prévision %r : la réduction est refusée" % (PREDICTED_SHRINK_OK,)
print("OK  prévision juste : réduction refusée")

**Résultat attendu** : `{0: 12, 1: 16, 2: 21, 3: 0, 4: 0, 5: 11}`. Les clients `C001`, `C003` et `C004` apparaissent maintenant dans deux partitions, 2 et 5. Le retour à 3 partitions est refusé : `The topic orders currently has 6 partition(s); 3 would not be an increase.`

### 6.8 Configurer le topic `clickstream`

Une configuration de topic (`retention.ms`, `retention.bytes`...) remplace, pour ce topic seulement, la valeur du cluster. `retention.bytes` s'applique **par partition**.

**Exemple en ligne de commande** : configurations propres à `orders`, puis liste des topics « avec configurations particulières ».

In [ ]:
!docker exec kafka-tools kafka-configs.sh --bootstrap-server $BS --describe --entity-type topics --entity-name orders
!docker exec kafka-tools kafka-topics.sh --bootstrap-server $BS --describe --topics-with-overrides

Équivalent en ligne de commande du TODO (**ne pas lancer en plus du TODO**) :

```bash
# [tools]
kafka-topics.sh --bootstrap-server $BS --create --topic clickstream --partitions 6 --replication-factor 3
kafka-configs.sh --bootstrap-server $BS --alter --entity-type topics --entity-name clickstream \
  --add-config retention.ms=86400000,retention.bytes=104857600
kafka-configs.sh --bootstrap-server $BS --describe --entity-type topics --entity-name clickstream
kafka-configs.sh --bootstrap-server $BS --alter --entity-type topics --entity-name clickstream \
  --delete-config retention.bytes
```

**Code fourni** : `alter()` pose ou retire des configurations d'un topic, comme `kafka-configs.sh --alter` ; `overrides()` lit les seules configurations propres au topic, comme `kafka-configs.sh --describe`.

In [ ]:
# Code fourni : poser, retirer et lire les configurations propres à un topic
def alter(topic, operation, **configs):
    """Applique une opération (SET ou DELETE) à des configurations du topic.

    Les noms sont écrits avec "_" au lieu de "." : retention_ms pour retention.ms.
    """
    admin = AdminClient(CONF)
    # Une entrée par configuration citée ; les autres configurations du topic ne sont pas touchées
    entries = [ConfigEntry(name.replace("_", "."), value, incremental_operation=operation)
               for name, value in configs.items()]
    resource = ConfigResource("topic", topic, incremental_configs=entries)
    for future in admin.incremental_alter_configs([resource]).values():
        future.result()


def overrides(topic):
    """{nom: valeur} des configurations propres au topic, sans les valeurs héritées du cluster."""
    admin = AdminClient(CONF)
    future = list(admin.describe_configs([ConfigResource("topic", topic)]).values())[0]
    # La source DYNAMIC_TOPIC_CONFIG désigne une configuration posée sur le topic lui-même
    return {name: e.value for name, e in future.result().items()
            if e.source == ConfigSource.DYNAMIC_TOPIC_CONFIG.value}

In [ ]:
# TODO 6.8 : remplacez les deux ... puis exécutez UNE SEULE FOIS
RETENTION_MS = ...      # rétention de clickstream : 24 heures, en millisecondes (un nombre)
REMOVE_OPERATION = ...  # opération qui retire une configuration du topic :
                        # AlterConfigOpType.SET ou AlterConfigOpType.DELETE ?

# Code fourni, à lire : rien à modifier en dessous
before_removal = {}
if filled(RETENTION_MS, REMOVE_OPERATION):
    admin = AdminClient(CONF)
    admin.create_topics([NewTopic("clickstream", num_partitions=6, replication_factor=3)])["clickstream"].result()
    # Rétention de 24 h et taille maximale de 100 Mo par partition (valeurs passées en texte)
    alter("clickstream", AlterConfigOpType.SET, retention_ms=str(RETENTION_MS), retention_bytes="104857600")
    before_removal = overrides("clickstream")
    # Retrait de retention.bytes : la valeur du cluster s'applique de nouveau
    alter("clickstream", REMOVE_OPERATION, retention_bytes=None)
    print("avant retrait :", before_removal)
    print("après retrait :", overrides("clickstream"))

In [ ]:
# Vérification 6.8
admin = AdminClient(CONF)
assert len(admin.list_topics("clickstream", timeout=10).topics["clickstream"].partitions) == 6, "À revoir : clickstream à 6 partitions"
print("OK  clickstream à 6 partitions")
assert before_removal == {"retention.ms": "86400000", "retention.bytes": "104857600"}, "À revoir : deux configurations posées : %s" % before_removal
print("OK  deux configurations posées : %s" % before_removal)
assert overrides("clickstream") == {"retention.ms": "86400000"}, "À revoir : retention.bytes retirée, retention.ms gardée"
print("OK  retention.bytes retirée, retention.ms gardée")
assert overrides("orders") == {}, "À revoir : orders n'a aucune configuration propre"
print("OK  orders n'a aucune configuration propre")

**Résultat attendu** : avant retrait, `retention.ms` et `retention.bytes` ; après, `retention.ms` seule. `orders` n'a aucune configuration propre, alors que `--topics-with-overrides` l'affiche quand même.

### 6.9 Produire sans clé

Sans clé, le producteur choisit lui-même la partition.

**Exemple en ligne de commande** : 1000 messages sans clé avec l'outil Java (cette commande est à exécuter : le `TODO` en ajoute 1000 autres avec le client Python, pour comparer).

In [ ]:
# L'outil Java attendrait 60 s par message si clickstream n'existait pas : on vérifie d'abord
admin = AdminClient(CONF)
if "clickstream" in admin.list_topics(timeout=10).topics:
    !docker exec kafka-tools bash -c "seq 1 1000 | sed 's/^/click-/' | kafka-console-producer.sh --bootstrap-server $BS --topic clickstream"
    !docker exec kafka-tools kafka-get-offsets.sh --bootstrap-server $BS --topic clickstream
else:
    print("clickstream absent : terminez d'abord l'étape 6.8")

In [ ]:
# TODO 6.9 : remplacez le ... par votre prévision, puis exécutez UNE SEULE FOIS
PREDICTED_EVEN_SPREAD = ...  # les 1000 messages sans clé seront-ils répartis à peu près également
                             # sur les 6 partitions, environ 167 chacune ? (True ou False)

# Code fourni, à lire : rien à modifier en dessous
distribution = {}
if filled(PREDICTED_EVEN_SPREAD):
    def count_delivered(err, msg):
        """Rapport de livraison : compte les messages écrits, partition par partition."""
        if err is None:
            distribution[msg.partition()] = distribution.get(msg.partition(), 0) + 1

    # "click-1" à "click-1000", sans clé : le producteur choisit lui-même la partition
    for i in range(1, 1001):
        producer.produce("clickstream", value="click-%d" % i, on_delivery=count_delivered)
    producer.flush(30)

print(distribution)

In [ ]:
# Vérification 6.9
assert sum(distribution.values()) == 1000, "À revoir : 1000 messages confirmés"
print("OK  1000 messages confirmés")
assert max(distribution.values()) >= 500, "À revoir : messages concentrés, pas répartis un par un : %s" % distribution
print("OK  messages concentrés, pas répartis un par un : %s" % distribution)
assert PREDICTED_EVEN_SPREAD is False, "À revoir : prévision %r : sans clé, les messages partent par lots dans une même partition" % (PREDICTED_EVEN_SPREAD,)
print("OK  prévision juste : répartition par lots, pas message par message")

**Résultat attendu** : les 1000 messages de l'outil Java se retrouvent tous, ou presque tous, dans une seule partition. Ceux du client Python sont aussi concentrés dans une partition, pas forcément la même ; selon la vitesse du poste, de 0 à une trentaine de messages partent dans chacune des autres partitions. Les partitions choisies varient d'un poste à l'autre.

### 6.10 Kafka UI et suppression d'un topic

Ouvrez http://localhost:8080, menu « Topics », topic `orders`, onglet « Messages » : retrouvez la partition, l'offset et la clé de quelques commandes.

La suppression d'un topic est asynchrone : le contrôleur l'enregistre, puis les brokers effacent les fichiers.

**Exemple en ligne de commande** :

In [ ]:
!docker exec kafka-tools kafka-topics.sh --bootstrap-server $BS --list

Équivalent en ligne de commande du TODO (**ne pas lancer en plus du TODO**) :

```bash
# [tools]
kafka-topics.sh --bootstrap-server $BS --create --topic tmp-test --partitions 1 --replication-factor 3
kafka-topics.sh --bootstrap-server $BS --delete --topic tmp-test
```

In [ ]:
# TODO 6.10 : remplacez le ... par votre prévision, puis exécutez
PREDICTED_IMMEDIATE = ...  # dès que delete_topics() a répondu, les fichiers du topic sont-ils déjà
                           # effacés sur tous les brokers ? (True ou False)

# Code fourni, à lire : rien à modifier en dessous
remaining_topics = []
if filled(PREDICTED_IMMEDIATE):
    admin = AdminClient(CONF)
    admin.create_topics([NewTopic("tmp-test", num_partitions=1, replication_factor=3)])["tmp-test"].result()
    # Le contrôleur enregistre la suppression et répond ; les brokers effacent ensuite les fichiers
    admin.delete_topics(["tmp-test"])["tmp-test"].result()
    # Attente : tmp-test ne figure plus dans la liste des topics
    wait_for(lambda: "tmp-test" not in admin.list_topics(timeout=10).topics, timeout=30,
             message="disparition de tmp-test")
    remaining_topics = sorted(admin.list_topics(timeout=10).topics)

print(remaining_topics)

In [ ]:
# Vérification 6.10
assert remaining_topics and "tmp-test" not in remaining_topics, "À revoir : tmp-test supprimé"
print("OK  tmp-test supprimé")
assert {"orders", "clickstream"} <= set(remaining_topics), "À revoir : orders et clickstream conservés pour le TP3"
print("OK  orders et clickstream conservés pour le TP3")
assert PREDICTED_IMMEDIATE is False, "À revoir : prévision %r : la suppression d'un topic est asynchrone" % (PREDICTED_IMMEDIATE,)
print("OK  prévision juste : suppression asynchrone")

**Résultat attendu** : `__consumer_offsets`, `clickstream` et `orders`, plus `loadgen` et `lab-validate` en mode `extended`. `tmp-test` n'apparaît plus.

## Fiche des topics livrés

La cellule suivante résume les topics livrés à NovaShop. Ces topics sont réutilisés au TP3 : ne les supprimez pas.

In [ ]:
# Code fourni : une ligne par topic livré, avec ses partitions, son RF, ses configurations et ses messages
admin = AdminClient(CONF)
sheet = []
for name in ("orders", "clickstream"):
    meta = admin.list_topics(name, timeout=10).topics[name]
    if meta.error is not None:  # topic absent : étape non faite
        sheet.append((name, "absent", "", "", ""))
        continue
    rf = len(meta.partitions[0].replicas)  # nombre de répliques de la partition 0
    configs = ", ".join("%s=%s" % kv for kv in sorted(overrides(name).items())) or "aucune"
    # Nombre de messages : somme des offsets de fin, lus par l'outil Java
    output = !docker exec kafka-tools kafka-get-offsets.sh --bootstrap-server $BS --topic {name} 2>&1
    messages = sum(int(n) for n in re.findall(r"^%s:\d+:(\d+)$" % name, output.n, re.M))
    sheet.append((name, len(meta.partitions), rf, configs, messages))

table(sheet, ["Topic", "Partitions", "RF", "Configurations propres", "Messages"])

**Attendu** : `orders`, 6 partitions, RF 3, aucune configuration propre, 60 messages ; `clickstream`, 6 partitions, RF 3, `retention.ms=86400000`, 2000 messages (1000 par l'outil Java, 1000 par le client Python).

## 7. Questions de réflexion

1. Pourquoi désactiver la création automatique des topics en production ?
2. Le facteur de réplication peut-il dépasser le nombre de brokers ? Et le nombre de partitions ?
3. Pourquoi Kafka refuse-t-il de réduire le nombre de partitions ?
4. Après l'ajout de partitions, l'ordre des commandes du client `C001` est-il encore garanti ? Que conseilleriez-vous à l'équipe Commandes ?
5. Pourquoi 1000 messages sans clé se retrouvent-ils dans une seule partition, alors qu'on parle souvent de répartition « round-robin » ?
6. Comment choisir le nombre de partitions d'un nouveau topic ?
7. Une application Java et une application Python écrivent dans `orders` avec la même clé client. Quel réglage faut-il vérifier, et pourquoi ?